# Shared Notebook 15/16 — certified frozen-feature evaluation

Purpose: evaluate frozen JEPA representations with a standardized regularized multinomial linear classifier stopped exclusively by training convergence, plus a fixed cosine kNN cross-check. Inputs: actual source-bound initial/final EMA checkpoints or verified Notebook14 feature caches. Outputs: full-state probe checkpoints, train-objective convergence certificates, predictions and paired development results. Runtime depends on convergence and endpoint count; every job honors the caller's absolute deadline. No experiment launches merely by loading this notebook.

Objective: mean training cross entropy plus `l2/2 × ||weights||²`; bias is unregularized. Default L2 is fixed at 1e−4. Training features determine mean/std. FP64 full-batch LBFGS uses no development-based epoch selection. Both gradient infinity norm and relative training-objective change must pass for three consecutive checks. Nonconvergence/cutoff remains UNKNOWN. This is a numerical certificate under declared tolerances, not exact optimization proof.

Probe checkpoints live under `checkpoints/<run_id>/stages/<stage_id>/convex_probes/`, or the saved cached-study checkpoint run ID. They retain latest two and the lowest training-objective checkpoint. Save every ten optimizer steps, at most 250 steps or 15 minutes, and at stop/convergence. Exact resume: call the identical `fit_convex_probe`/`evaluate_stage`/`cached14_study` function with identical source/config/cache IDs; invalid manifests fail and never silently restart. Labels enter probe/kNN evaluation only. Official-test pixels are never opened by these development APIs.

## Source, objective and checkpoint contracts

In [ ]:
"""Shared Notebook15/16 evaluation. Tagged definitions only; no automatic experiment."""
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG",":4096:8")
from pathlib import Path
import copy,csv,hashlib,json,math,random,time,types,sys
import nbformat
import numpy as np
import torch
from scipy.stats import t as student_t
ROOT=Path(os.environ.get("FMM_JEPA_ROOT","/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
SOURCE=Path(globals().get("__file__",str(ROOT/"experiments/15_vit_geometry/evaluation.ipynb"))).resolve()
_boot={"__file__":str(ROOT/"fmm_jepa/bootstrap.ipynb")}
for _cell in nbformat.read(ROOT/"fmm_jepa/bootstrap.ipynb",as_version=4).cells:
    if _cell.cell_type=="code" and "ahr-export"in _cell.metadata.get("tags",[]):exec(compile(_cell.source,"bootstrap#ahr-export","exec"),_boot)
_boot["install_notebook_package"](ROOT)
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.training import atomic_json,digest_file,run_writer
from fmm_jepa.food101_evaluation import (atomic_torch,encoder_state_hash,validate_feature_cache,extract_features,embedding_diagnostics,write_csv)
from fmm_jepa.food101_data import EvaluationDataset,preprocessing_descriptor
DEFAULT_PROBE={"schema":"standardized-convex-linear/v1","num_classes":101,"l2":1e-4,"max_steps":1000,
    "gradient_inf_tolerance":1e-5,"relative_objective_tolerance":1e-9,"stable_checks":3,
    "history_size":50,"lr":1.,"checkpoint_every":10,"standardization_epsilon":1e-6,"cpu_threads":2,
    "dtype":"float64","optimizer":"torch_LBFGS_full_batch_strong_wolfe","knn_k":20,"knn_query_chunk":128,"knn_train_chunk":2048}
_BUILDER=None
_BUILDER_IDENTITY=None


def _canonical(value):return json.loads(json.dumps(value,sort_keys=True,allow_nan=False))
def _hash(value):return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()
def _read(path):return json.loads(Path(path).read_text())
def _record(path):return {"path":str(Path(path).resolve()),"sha256":digest_file(path)}
def _verify_record(record,root=None):
    path=Path(record["path"]).resolve()
    if root is not None and not path.is_relative_to(Path(root).resolve()):raise ValueError("Evidence path outside project")
    if digest_file(path)!=record["sha256"]:raise ValueError("Evidence checksum differs: "+str(path))
    return path

def export_source_hash():
    sources=[c.source for c in nbformat.read(SOURCE,as_version=4).cells if c.cell_type=="code" and "ahr-export"in c.metadata.get("tags",[])]
    return hashlib.sha256("\n".join(sources).encode()).hexdigest()

def probe_configuration(overrides=None):
    value=copy.deepcopy(DEFAULT_PROBE)
    if overrides:value.update(overrides)
    if value["optimizer"]!="torch_LBFGS_full_batch_strong_wolfe"or value["dtype"]!="float64":raise ValueError("Only the registered deterministic FP64 full-batch solver is supported")
    if not math.isfinite(float(value["l2"]))or not 0<float(value["l2"]) or int(value["max_steps"])<1 or int(value["num_classes"])<2:raise ValueError("Positive L2/budget and at least two classes required")
    if not 1<=int(value["checkpoint_every"])<=250 or int(value["stable_checks"])<1:raise ValueError("Checkpoint at most every250 optimizer steps")
    for name in ("gradient_inf_tolerance","relative_objective_tolerance","standardization_epsilon"):
        if not math.isfinite(float(value[name]))or float(value[name])<=0:raise ValueError("Finite positive convergence/standardization settings required")
    for name in ("lr",):
        if not math.isfinite(float(value[name]))or float(value[name])<=0:raise ValueError("Finite positive solver learning rate required")
    for name in ("history_size","cpu_threads","knn_k","knn_query_chunk","knn_train_chunk"):
        if int(value[name])<1:raise ValueError("Positive solver and kNN dimensions required")
    return value


def _rng():return {"python":random.getstate(),"numpy":np.random.get_state(),"torch_cpu":torch.get_rng_state(),"torch_cuda":torch.cuda.get_rng_state_all()if torch.cuda.is_available()else []}
def _restore_rng(value):
    random.setstate(value["python"]);np.random.set_state(value["numpy"]);torch.set_rng_state(value["torch_cpu"])
    if value["torch_cuda"]:torch.cuda.set_rng_state_all(value["torch_cuda"])


def _validate_inputs(train,query,classes):
    for cache in (train,query):
        x,y,ids=cache["features"],cache["labels"],cache["sample_ids"]
        if x.ndim!=2 or y.ndim!=1 or len(ids)!=len(x)or len(y)!=len(x)or len(ids)!=len(set(ids))or not len(ids):raise ValueError("Nonempty unique aligned feature/label/ID arrays required")
        if x.device.type!="cpu"or y.device.type!="cpu"or not torch.isfinite(x).all()or y.dtype!=torch.long:raise ValueError("Finite CPU features and integer labels required")
        if bool(((y<0)|(y>=classes)).any()):raise ValueError("Labels outside declared classes")
    if train["features"].shape[1]!=query["features"].shape[1]or set(train["sample_ids"])&set(query["sample_ids"]):raise ValueError("Feature dimensions differ or memberships overlap")
    if set(train["labels"].tolist())!=set(range(classes)):raise ValueError("Every declared class must be represented in probe training")


def _identity(train,config,model_id):
    return {"schema":"notebook15-convex-probe-identity/v1","model_id":model_id,"config":config,
        "training_content_hash":encoder_state_hash({"features":train["features"],"labels":train["labels"]}),
        "training_sample_ids":list(train["sample_ids"]),"training_cache_identity":_canonical(train.get("identity",{})),
        "evaluation_export_hash":export_source_hash(),"torch_version":torch.__version__,"precision":"CPU float64 full batch",
        "objective":"Mean training multinomial cross entropy + (l2/2)*sum(weight**2); bias unregularized",
        "selection":"Training gradient and objective criteria only; no development epoch/score selection"}


def _load_probe(directory,identity=None):
    directory=Path(directory);manifest=_read(directory/"manifest.json")
    if manifest.get("schema")!="notebook15-convex-probe-manifest/v1":raise ValueError("Invalid convex probe checkpoint schema")
    if identity is not None and manifest["identity"]!=identity:raise ValueError("Probe identity/config/cache/source changed; refusing restart")
    entries=manifest["checkpoints"]
    if len(entries)<1 or manifest["latest"]not in {e["file"]for e in entries}or manifest["best"]not in {e["file"]for e in entries}:raise ValueError("Invalid retained probe checkpoints")
    for record in entries:
        if Path(record["file"]).name!=record["file"]or digest_file(directory/record["file"])!=record["sha256"]:raise ValueError("Convex probe checkpoint hash mismatch")
    latest=next(e for e in entries if e["file"]==manifest["latest"])
    state=torch.load(directory/latest["file"],map_location="cpu",weights_only=False)
    if state["identity"]!=manifest["identity"]or int(state["step"])!=int(latest["step"]):raise ValueError("Probe state and manifest disagree")
    return state,manifest


def _save_probe(directory,state):
    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)
    path=directory/("step_%06d.pt"%state["step"]);atomic_torch(path,state)
    manifest_path=directory/"manifest.json"
    old=_read(manifest_path)if manifest_path.exists()else None
    entries=copy.deepcopy(old["checkpoints"])if old else []
    record={"file":path.name,"sha256":digest_file(path),"step":state["step"],"train_objective":state["metrics"][-1]["objective"]}
    entries=[e for e in entries if e["file"]!=path.name]+[record]
    ordered=sorted(entries,key=lambda e:e["step"]);best=min(entries,key=lambda e:(e["train_objective"],e["step"]))
    retained={e["file"]for e in ordered[-2:]}|{best["file"]};keep=[e for e in ordered if e["file"]in retained]
    manifest={"schema":"notebook15-convex-probe-manifest/v1","identity":state["identity"],"latest":path.name,"best":best["file"],"checkpoints":keep,
        "status":state["status"],"step":state["step"],"resume_notebook":"notebooks/15_ViT_Equivalent_Geometry.ipynb","resume_function":"fit_convex_probe with identical train cache/config/model_id/directory; latest valid checkpoint is mandatory",
        "best_policy":"Lowest training objective among saved checkpoints; no development-based selection","checkpoint_policy":"Every10 steps by default, at most250 or15min, boundary/interruption; retain latest2 plus best"}
    atomic_json(manifest_path,manifest)
    for e in entries:
        if e["file"]not in retained:
            p=directory/e["file"]
            if p.exists():p.unlink()
    return manifest


## Converged convex readout and deterministic kNN

In [ ]:
def fit_convex_probe(train_cache,query_cache,directory,config=None,model_id="probe",deadline_unix=None,stop_after_steps=None):
    """FP64 full-batch convex readout; return UNKNOWN unless fixed training certificate passes."""
    config=probe_configuration(config);classes=int(config["num_classes"]);_validate_inputs(train_cache,query_cache,classes)
    identity=_identity(train_cache,config,model_id);directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)
    caller_rng=_rng();threads=torch.get_num_threads();torch.set_num_threads(int(config["cpu_threads"]))
    try:
        with run_writer(directory/"writer_lock"):
            model=torch.nn.Linear(train_cache["features"].shape[1],classes,dtype=torch.float64)
            with torch.no_grad():model.weight.zero_();model.bias.zero_()
            optimizer=torch.optim.LBFGS(model.parameters(),lr=float(config["lr"]),max_iter=1,max_eval=25,
                tolerance_grad=0.,tolerance_change=0.,history_size=int(config["history_size"]),line_search_fn="strong_wolfe")
            manifest_path=directory/"manifest.json"
            if manifest_path.exists():
                state,manifest=_load_probe(directory,identity);model.load_state_dict(state["model"]);optimizer.load_state_dict(state["optimizer"]);_restore_rng(state["rng"])
            else:
                if list(directory.glob("step_*.pt")):raise ValueError("Orphan probe checkpoint; never silently restart")
                x=train_cache["features"].double();mean=x.mean(0);std=x.std(0,unbiased=False).clamp_min(float(config["standardization_epsilon"]))
                state={"schema":"notebook15-convex-probe-state/v1","identity":identity,"model":None,"optimizer":None,"scheduler":None,"scaler":None,
                    "standardization":{"mean":mean,"std":std,"fit_scope":"probe_train_only","fit_examples":len(x)},"rng":None,"precision":"CPU float64",
                    "sampler":{"mode":"full_batch","sample_ids":list(train_cache["sample_ids"]),"completed_full_batches":0},"step":0,"metrics":[],"stable_checks":0,
                    "function_evaluations":0,"status":"RUNNING","converged":False}
            x=(train_cache["features"].double()-state["standardization"]["mean"])/state["standardization"]["std"];y=train_cache["labels"]
            def closure():
                optimizer.zero_grad(set_to_none=True)
                objective=torch.nn.functional.cross_entropy(model(x),y)+.5*float(config["l2"])*model.weight.square().sum()
                if not torch.isfinite(objective):raise FloatingPointError("Nonfinite convex training objective")
                objective.backward();state["function_evaluations"]+=1
                if not all(p.grad is not None and torch.isfinite(p.grad).all()for p in model.parameters()):raise FloatingPointError("Nonfinite classifier gradient")
                return objective
            def measure():
                objective=float(closure().detach());grad=max(float(p.grad.abs().max())for p in model.parameters())
                previous=state["metrics"][-1]["objective"]if state["metrics"]else None
                relative=None if previous is None else abs(objective-previous)/max(1.,abs(previous))
                return {"step":state["step"],"objective":objective,"gradient_inf_norm":grad,"relative_objective_change":relative}
            def save(status):
                state.update(model=copy.deepcopy(model.state_dict()),optimizer=copy.deepcopy(optimizer.state_dict()),rng=_rng(),status=status)
                state["sampler"]["completed_full_batches"]=state["step"]
                return _save_probe(directory,state)
            if not state["metrics"]:state["metrics"].append(measure());manifest=save("RUNNING")
            max_steps=int(config["max_steps"]);limit=min(max_steps,int(stop_after_steps))if stop_after_steps is not None else max_steps
            last_save=time.monotonic();cutoff=False
            while state["step"]<limit and not state["converged"]:
                if deadline_unix is not None and time.time()+1>=float(deadline_unix):cutoff=True;break
                optimizer.step(closure);state["step"]+=1;metric=measure();state["metrics"].append(metric)
                criterion=metric["gradient_inf_norm"]<=float(config["gradient_inf_tolerance"])and metric["relative_objective_change"]is not None and metric["relative_objective_change"]<=float(config["relative_objective_tolerance"])
                state["stable_checks"]=state["stable_checks"]+1 if criterion else 0
                state["converged"]=state["stable_checks"]>=int(config["stable_checks"])
                if state["step"]%int(config["checkpoint_every"])==0 or state["converged"]or time.monotonic()-last_save>=900:
                    manifest=save("CONVERGED"if state["converged"]else"RUNNING");last_save=time.monotonic()
            status="CONVERGED"if state["converged"]else"UNKNOWN_CUTOFF"if cutoff else"INTERRUPTED"if stop_after_steps is not None and state["step"]<max_steps else"UNKNOWN_NOT_CONVERGED"
            manifest=save(status)
            receipt={"status":status,"converged":bool(state["converged"]),"model_id":model_id,"optimizer_steps":state["step"],"function_evaluations":state["function_evaluations"],
                "train_objective":state["metrics"][-1]["objective"],"gradient_inf_norm":state["metrics"][-1]["gradient_inf_norm"],"relative_objective_change":state["metrics"][-1]["relative_objective_change"],
                "stable_checks":state["stable_checks"],"config":config,"identity_hash":_hash(identity),"manifest":_record(manifest_path),"latest_checkpoint":_record(directory/manifest["latest"]),
                "query_identity_hash":_hash({"sample_ids":query_cache["sample_ids"],"content":encoder_state_hash({"features":query_cache["features"],"labels":query_cache["labels"]})}),"certificate":"Training grad infinity norm and relative objective change pass for fixed consecutive checks; no development objective/score used for stopping",
                "query_metrics":None,"predictions":None,"training_metrics":copy.deepcopy(state["metrics"])}
            if state["converged"]:
                with torch.inference_mode():
                    z=(query_cache["features"].double()-state["standardization"]["mean"])/state["standardization"]["std"]
                    logits=model(z);pred=logits.argmax(-1);score=float((pred==query_cache["labels"]).double().mean())
                    receipt["query_metrics"]={"top1":score,"loss":float(torch.nn.functional.cross_entropy(logits,query_cache["labels"])),"examples":len(z)}
                    receipt["predictions"]=[{"sample_id":sample_id,"label":int(label),"top1_prediction":int(prediction)}for sample_id,label,prediction in zip(query_cache["sample_ids"],query_cache["labels"].tolist(),pred.tolist())]
            atomic_json(directory/"probe_receipt.json",{k:v for k,v in receipt.items()if k not in("training_metrics","predictions")})
            receipt["receipt"]=_record(directory/"probe_receipt.json");return receipt
    finally:_restore_rng(caller_rng);torch.set_num_threads(threads)


def _cosine_knn(train_cache,query_cache,config=None,deadline_unix=None):
    """Fixed k=20 cosine majority vote; equal similarities favor earlier train IDs, tied votes lower class."""
    config=probe_configuration(config);_validate_inputs(train_cache,query_cache,int(config["num_classes"]))
    k=min(int(config["knn_k"]),len(train_cache["features"]));qchunk=int(config["knn_query_chunk"]);tchunk=int(config["knn_train_chunk"])
    if k<1 or qchunk<1 or tchunk<1:raise ValueError("Positive k and chunk sizes required")
    x=torch.nn.functional.normalize(train_cache["features"].double(),dim=1,eps=1e-12);q=torch.nn.functional.normalize(query_cache["features"].double(),dim=1,eps=1e-12)
    predictions=[];classes=int(config["num_classes"])
    for start in range(0,len(q),qchunk):
        if deadline_unix is not None and time.time()>=float(deadline_unix):return {"status":"UNKNOWN_CUTOFF","predictions":None}
        query=q[start:start+qchunk];best_scores=torch.empty((len(query),0),dtype=torch.float64);best_ids=torch.empty((len(query),0),dtype=torch.long)
        for offset in range(0,len(x),tchunk):
            if deadline_unix is not None and time.time()>=float(deadline_unix):return {"status":"UNKNOWN_CUTOFF","predictions":None}
            scores=query@x[offset:offset+tchunk].T;ids=torch.arange(offset,min(offset+tchunk,len(x))).expand(len(query),-1)
            scores=torch.cat((best_scores,scores),1);ids=torch.cat((best_ids,ids),1)
            by_id=torch.argsort(ids,dim=1,stable=True);scores=scores.gather(1,by_id);ids=ids.gather(1,by_id)
            selected=torch.argsort(scores,dim=1,descending=True,stable=True)[:,:k];best_scores=scores.gather(1,selected);best_ids=ids.gather(1,selected)
        neighbor_labels=train_cache["labels"][best_ids];votes=torch.zeros((len(query),classes),dtype=torch.long)
        votes.scatter_add_(1,neighbor_labels,torch.ones_like(neighbor_labels));predictions.extend(votes.argmax(1).tolist())
    pred=torch.tensor(predictions);return {"status":"COMPLETE","k":k,"accuracy_percent":100*float((pred==query_cache["labels"]).double().mean()),
        "predictions":predictions,"normalization":"Raw frozen image embeddings normalized to unit cosine; no label-dependent preprocessing","tie_rule":"Source index then class index; stable across chunk sizes"}


def _paired(values,seeds):
    values=np.asarray(values,dtype=float);n=len(values)
    if n<2 or len(set(seeds))!=n or not np.isfinite(values).all():raise ValueError("Distinct finite paired seed effects required")
    mean=float(values.mean());sd=float(values.std(ddof=1));se=sd/math.sqrt(n);radius=float(student_t.ppf(.975,n-1))*se
    return {"n_pairs":n,"degrees_of_freedom":n-1,"mean_percentage_points":mean,"sd_percentage_points":sd,"raw_t_ci95_percentage_points":[mean-radius,mean+radius]if se>0 else None,
        "paired_t_two_sided_p_value":float(2*student_t.sf(abs(mean/se),n-1))if se>0 else 1. if mean==0 else None,
        "positive_pairs":int((values>0).sum()),"paired_seed_deltas":[{"seed":int(s),"delta_percentage_points":float(v)}for s,v in zip(seeds,values)]}


def cosine_knn(train_cache,query_cache,config=None,deadline_unix=None):
    config=probe_configuration(config);threads=torch.get_num_threads();torch.set_num_threads(int(config["cpu_threads"]))
    try:return _cosine_knn(train_cache,query_cache,config,deadline_unix)
    finally:torch.set_num_threads(threads)


## Source-bound initial/final stages and cached Notebook14 readout

In [ ]:
def bind_model_builder(builder,source_identity):
    global _BUILDER,_BUILDER_IDENTITY
    if not callable(builder)or not isinstance(source_identity,dict)or not source_identity:raise ValueError("Callable builder and immutable source identity required")
    _verify_builder_identity(ROOT,source_identity)
    _BUILDER=builder;_BUILDER_IDENTITY=_canonical(source_identity)


def _verify_builder_identity(root,identity):
    sources=identity.get("source_hashes",identity)
    for relative,expected in sources.items():
        path=(Path(root)/relative).resolve()
        if not path.is_relative_to(Path(root).resolve())or path.suffix!=".ipynb"or not isinstance(expected,str):raise ValueError("Builder must bind project notebook export hashes")
        cells=nbformat.read(path,4).cells;actual=hashlib.sha256("\n".join(cell.source for cell in cells if cell.cell_type=="code"and"ahr-export"in cell.metadata.get("tags",[])).encode()).hexdigest()
        if actual!=expected:raise ValueError("Bound evaluation builder source changed: "+relative)
    return sources


def _builder(root):
    if _BUILDER is not None:
        _verify_builder_identity(root,_BUILDER_IDENTITY);return _BUILDER,_BUILDER_IDENTITY
    path=Path(root)/"experiments/15_vit_geometry/runner.ipynb";name="fmm15_eval_model_builder"
    if name not in sys.modules:
        module=types.ModuleType(name);module.__file__=str(path);TaggedNotebookLoader(path).exec_module(module);sys.modules[name]=module
    module=sys.modules[name]
    sources=module.stage_definition_hashes(root)
    if not hasattr(module,"_evaluation_loaded_hashes"):module._evaluation_loaded_hashes=copy.deepcopy(sources)
    if sources!=module._evaluation_loaded_hashes:raise ValueError("Previously loaded builder source changed; restart kernel")
    return module.build_model,{"notebook":str(path),"source_hashes":sources}


def _deadline(deadline):
    if deadline is not None and time.time()>=float(deadline):raise TimeoutError("Evaluation cutoff; missing outcomes remain UNKNOWN")

class _DeadlineDataset:
    def __init__(self,dataset,deadline):self.dataset,self.deadline=dataset,deadline
    def __len__(self):return len(self.dataset)
    def __getitem__(self,index):_deadline(self.deadline);return self.dataset[index]


def _cache_from_record(root,record):
    path=_verify_record(record,root);metadata_path=path.with_suffix(".json")
    if not metadata_path.exists():raise ValueError("Feature-cache metadata missing")
    metadata=_read(metadata_path)
    if metadata["sha256"]!=digest_file(path):raise ValueError("Cache metadata checksum differs")
    cache=torch.load(path,map_location="cpu",weights_only=True);validate_feature_cache(cache,metadata["identity"])
    return {**cache,"path":str(path),"metadata":_record(metadata_path)}


def _diagnostics(train,dev):
    count=min(202,len(dev["features"]));features=dev["features"][:count]
    return {"finite_train_and_development":bool(torch.isfinite(train["features"]).all()and torch.isfinite(dev["features"]).all()),
        "training_representation_not_exactly_constant":bool((train["features"]!=train["features"][0]).any()),"development_representation_not_exactly_constant":bool((dev["features"]!=dev["features"][0]).any()),
        "descriptive":embedding_diagnostics(features),"scope":"First202 development embeddings before standardization; effective rank descriptive only"}


def _evaluate_caches(root,artifact,probe_dir,model_id,arm,endpoint,step,train,dev,source,config,deadline,training_cost=None):
    _deadline(deadline);diagnostic=_diagnostics(train,dev);valid=all(diagnostic[k]for k in("finite_train_and_development","training_representation_not_exactly_constant","development_representation_not_exactly_constant"))
    probe=fit_convex_probe(train,dev,probe_dir,config,model_id,deadline_unix=deadline)
    knn=cosine_knn(train,dev,config,deadline_unix=deadline)if probe["converged"]else{"status":"UNKNOWN_PROBE_NOT_CONVERGED"}
    accuracy=100*probe["query_metrics"]["top1"]if probe["converged"]else None
    summary={"status":probe["status"],"converged":probe["converged"],"metrics":probe["query_metrics"],"selection":"certified_train_objective_convergence",
        "standardization_fit_scope":"probe_train_only","encoder_unchanged":True,"encoder_scope":"Hash-verified frozen extraction or existing committed cache; no encoder optimizer","config":config,"receipt":probe["receipt"]}
    result={"model_id":model_id,"arm_id":arm["id"],"seed":int(arm["seed"]),"variant":arm["variant"],"kind":arm.get("kind"),"endpoint":endpoint,"updates":int(step),
        "status":"COMPLETE"if probe["converged"]and knn.get("status")=="COMPLETE"else"UNKNOWN_INCOMPLETE_OR_NONCONVERGED","accuracy_percent":accuracy,"knn_accuracy_percent":knn.get("accuracy_percent"),"converged":bool(probe["converged"]),"numerical_valid":valid,"probe_receipt":probe["receipt"],
        "probe":{"summary":summary,"convergence_certificate":{k:v for k,v in probe.items()if k not in("predictions","training_metrics")},"metrics":probe["training_metrics"]},
        "knn":{k:v for k,v in knn.items()if k!="predictions"},"source_encoder":source,"feature_caches":{"train":_record(train["path"]),"development":_record(dev["path"])},
        "representation_diagnostics":diagnostic,"training_cost":training_cost,"heldout_status":"NOT_RUN"}
    if probe["converged"]:
        path=Path(artifact)/"predictions"/(model_id+".csv");path.parent.mkdir(parents=True,exist_ok=True)
        write_csv(path,probe["predictions"])
        result["prediction_record"]=_record(path)
    return result


def _summary(evaluations,plan):
    rows=[];index={(value["variant"],int(value["seed"]),int(value["updates"])):value for value in evaluations.values()};final=int(plan["updates"]);seeds=list(map(int,plan["seeds"]));variants=list(plan["variants"])
    for value in evaluations.values():
        initial=index.get((value["variant"],int(value["seed"]),0));gain=value["accuracy_percent"]-initial["accuracy_percent"]if initial and initial["accuracy_percent"]is not None and value["accuracy_percent"]is not None else None
        cost=value.get("training_cost")or {};perf=cost.get("performance")or {}
        rows.append({"model_id":value["model_id"],"variant":value["variant"],"seed":value["seed"],"updates":value["updates"],"endpoint":value["endpoint"],"top1_percent":value["accuracy_percent"],"accuracy_percent":value["accuracy_percent"],"knn_accuracy_percent":value["knn_accuracy_percent"],
            "initial_top1_percent":initial["accuracy_percent"]if initial else None,"gain_percentage_points":gain,"converged":value["converged"],"numerical_valid":value["numerical_valid"],
            "median_training_step_seconds":(perf.get("complete_jepa_step_seconds")or{}).get("median"),"peak_training_GiB":perf["peak_cuda_allocated_bytes"]/1024**3 if perf.get("peak_cuda_allocated_bytes")is not None else None})
    contrasts={}
    if "vit"in variants:
        for variant in variants:
            if variant=="vit":continue
            if all((variant,s,final)in index and("vit",s,final)in index and index[(variant,s,final)]["converged"]and index[("vit",s,final)]["converged"]for s in seeds):
                contrasts[variant+"_minus_vit_final"]=_paired([index[(variant,s,final)]["accuracy_percent"]-index[("vit",s,final)]["accuracy_percent"]for s in seeds],seeds)
    return {"chart_rows":rows,"contrasts":contrasts,"primary_analysis":{"role":"Registered development study; final-test confirmation requires separate sealed reader","comparisons":contrasts,"n_pairs":len(seeds)},
        "learning_gate":{"passed":all(v["numerical_valid"]and v["converged"]for v in evaluations.values()),"status":"NUMERICAL_AND_CERTIFIED_READOUT_ONLY"}}


def evaluate_stage(root,artifact,plan,bundle,training,deadline_unix):
    """Evaluate all declared initial/final frozen EMA endpoints; no official test access."""
    root,artifact=Path(root).resolve(),Path(artifact).resolve()
    if not artifact.is_relative_to(root):raise ValueError("Evaluation artifact outside project")
    if training.get("status")!="COMPLETE"or not all(training.get(k)for k in("equal_updates","paired_traces_within_seed","no_failed_attempts")):raise ValueError("Clean fully paired completed training required")
    if set(training["arms"])!={a["id"]for a in plan["arms"]}:raise ValueError("Training arm IDs differ")
    builder,builder_identity=_builder(root);config=probe_configuration(plan.get("probe_convex"));artifact.mkdir(parents=True,exist_ok=True)
    source_hashes=builder_identity.get("source_hashes",builder_identity)
    if plan.get("source_hashes") is not None and source_hashes!=plan["source_hashes"]:raise ValueError("Evaluation builder sources differ from registered training sources")
    memberships=[set(bundle["probe_train_ids"]),set(bundle["dev_ids"]),set(bundle.get("heldout_ids",[]))]
    if any(memberships[i]&memberships[j]for i in range(3)for j in range(i+1,3)):raise ValueError("Evaluation development/official memberships overlap")
    jobs=[]
    for arm in plan["arms"]:
        if arm["config"].get("evaluation",{}).get("heldout_enabled",False):raise ValueError("Official test remains closed in development evaluate_stage")
        record=training["arms"][arm["id"]]
        if record["successful_updates"]!=plan["updates"]or record["attempted_updates"]!=plan["updates"]or record["failures"]:raise ValueError("Incomplete or failed training arm")
        if "initial"in plan.get("probe_endpoints",[]):jobs.append((arm,"initial",0,record["initial_teacher"],None))
        jobs.append((arm,"final",int(plan["updates"]),record["milestones"][str(plan["updates"])],record["milestones"][str(plan["updates"])]))
    identity={"schema":"notebook15-certified-stage/v1","plan":_canonical(plan),"builder":builder_identity,"evaluation_export_hash":export_source_hash(),"probe_config":config,
        "dataset_fingerprint":bundle["dataset"].fingerprint,"training_ids":list(bundle["probe_train_ids"]),"development_ids":list(bundle["dev_ids"])}
    binding_path=artifact/"evaluation_registration.json"
    if binding_path.exists():
        if _read(binding_path)!=identity:raise ValueError("Evaluation plan/config/source/membership identity changed")
    else:atomic_json(binding_path,identity)
    with run_writer(artifact/"evaluation_writer"):
        evaluations={};started=time.time()
        try:
            for arm,endpoint,step,source,cost in jobs:
                _deadline(deadline_unix);model_id=arm["id"]+("-initial"if endpoint=="initial"else"-u"+str(step));path=artifact/"evaluation"/(model_id+".json")
                if path.exists():
                    value=_read(path)
                    if value.get("evaluation_identity_hash")!=_hash(identity):raise ValueError("Saved evaluation identity differs")
                    _verify_record(value["source_encoder"],root);_verify_record(value["probe_receipt"],root)
                    if value.get("prediction_record"):_verify_record(value["prediction_record"],root)
                    for cache_record in value["feature_caches"].values():_cache_from_record(root,cache_record)
                    if value["converged"]and value.get("knn",{}).get("status")=="COMPLETE":evaluations[model_id]=value;continue
                _verify_record(source,root);model=builder(copy.deepcopy(arm["config"]),arm["kind"]).cpu()
                state=torch.load(source["path"],map_location="cpu",weights_only=(step==0))
                if step==0:model.target.load_state_dict(state)
                else:
                    if int(state["state"]["successful_updates"])!=step or state["config"]!=arm["config"]:raise ValueError("Evaluated model schedule/config differs")
                    model.load_state_dict(state["model"])
                encoder=model.target;del state,model
                common={"dataset_revision":bundle["dataset"].fingerprint,"checkpoint_hash":encoder_state_hash(encoder),"encoder_kind":getattr(encoder,"kind",arm["kind"]),
                    "preprocessing":preprocessing_descriptor(arm["config"]),"pooling":"mean_final_layernorm_patches","normalization":"Final LayerNorm patch mean; probe standardization fit on training features only"}
                caches={}
                for split,ids in(("train",bundle["probe_train_ids"]),("development",bundle["dev_ids"])):
                    dataset=_DeadlineDataset(EvaluationDataset(bundle["dataset"],ids,arm["config"]),deadline_unix)
                    caches[split]=extract_features(encoder,dataset,artifact/"features"/model_id,{**common,"sample_ids":list(ids)},batch_size=int(arm["config"].get("evaluation",{}).get("batch_size",64)),device=arm["config"]["device"])
                encoder.cpu();del encoder
                if torch.cuda.is_available():torch.cuda.empty_cache()
                probe_dir=root/"checkpoints"/plan["run_id"]/"stages"/plan["stage_id"]/"convex_probes"/model_id
                value=_evaluate_caches(root,artifact,probe_dir,model_id,arm,endpoint,step,caches["train"],caches["development"],source,config,deadline_unix,cost)
                value["evaluation_identity_hash"]=_hash(identity);atomic_json(path,value);evaluations[model_id]=value
                atomic_json(artifact/"evaluation_status.json",{"status":"RUNNING","completed_models":sum(v.get("converged",False)for v in evaluations.values()),"total_models":len(jobs),"current_model":model_id,"updated_unix":time.time()})
                if value.get("status")!="COMPLETE":break
        except TimeoutError as error:
            atomic_json(artifact/"evaluation_status.json",{"status":"UNKNOWN_CUTOFF","error":repr(error),"completed_models":len(evaluations),"total_models":len(jobs),"updated_unix":time.time()})
        complete=len(evaluations)==len(jobs)and all(v.get("converged")and v.get("knn",{}).get("status")=="COMPLETE"for v in evaluations.values())
        result={"schema":"notebook15-converged-stage-evaluation/v1","status":"COMPLETE"if complete else"UNKNOWN_INCOMPLETE_OR_NONCONVERGED","evaluations":evaluations,
            "run_id":plan["run_id"],"stage_id":plan["stage_id"],"plan_hash":_hash(plan),"probe_configuration":config,"probe_selection":"certified_training_objective_convergence",
            "started_unix":started,"completed_unix":time.time(),"official_test":"NOT_RUN","labels_in_self_supervised_loss":False,"builder_identity":builder_identity}
        if complete:result.update(_summary(evaluations,plan))
        else:result.update(chart_rows=[],contrasts={},learning_gate={"passed":False,"status":"UNKNOWN"})
        atomic_json(artifact/"stage_evaluation.json",result);return result


def cached14_study(root,artifact,deadline_unix):
    """Actual16 final caches only. Legacy epoch20 scores retained for readout comparison; no new JEPA."""
    root,artifact=Path(root).resolve(),Path(artifact).resolve()
    if not artifact.is_relative_to(root):raise ValueError("Cached study artifact outside project")
    source_path=root/"artifacts/notebook_14/routing-capacity-001/stage_evaluation.json";saved=_read(source_path)
    if saved.get("status")!="COMPLETE"or len(saved["evaluations"])!=16:raise ValueError("Actual complete16 Notebook14 final encoders required")
    evidence={}
    for old in saved["evaluations"].values():
        evidence[old["model_id"]]={"source_encoder":old["source_encoder"],"feature_caches":old["feature_caches"],"legacy_predictions":_record(old["probe"]["summary"]["predictions_path"])}
    config=probe_configuration();identity={"schema":"notebook15-cached14-readout/v1","source_evidence":evidence,"source":_record(source_path),"evaluation_export_hash":export_source_hash(),"config":config,
        "scope":"All16 actual registered final600 encoders; no matched scientific initialization caches, no new model training, reused development labels"}
    artifact.mkdir(parents=True,exist_ok=True);registration=artifact/"registration.json"
    if registration.exists():
        if _read(registration)!=identity:raise ValueError("Cached readout identity changed")
    else:atomic_json(registration,identity)
    evaluations={};checkpoint_run=artifact.name
    with run_writer(artifact/"writer_lock"):
        for old in saved["evaluations"].values():
            try:_deadline(deadline_unix)
            except TimeoutError:break
            if int(old["updates"])!=600:raise ValueError("Only actual final600 caches enter this study")
            model_id=old["model_id"];recordpath=artifact/"evaluation"/(model_id+".json")
            if recordpath.exists():
                value=_read(recordpath)
                if value.get("cached_identity_hash")!=_hash(identity):raise ValueError("Cached endpoint identity differs")
                _verify_record(value["probe_receipt"],root);_verify_record(value["source_encoder"],root);_verify_record(value["legacy_predictions"],root)
                if value.get("prediction_record"):_verify_record(value["prediction_record"],root)
                for cache_record in value["feature_caches"].values():_cache_from_record(root,cache_record)
                if value.get("converged")and value.get("knn",{}).get("status")=="COMPLETE":evaluations[model_id]=value;continue
            train=_cache_from_record(root,old["feature_caches"]["train"]);dev=_cache_from_record(root,old["feature_caches"]["development"])
            if train["identity"]["checkpoint_hash"]!=old["probe"]["summary"]["checkpoint_hash"]or dev["identity"]["checkpoint_hash"]!=train["identity"]["checkpoint_hash"]:raise ValueError("Cached encoder hashes differ from actual endpoint")
            _verify_record(old["source_encoder"],root)
            legacy=Path(old["probe"]["summary"]["predictions_path"])
            if old["probe"]["summary"].get("selected_epoch")!=20 or not legacy.resolve().is_relative_to(root):raise ValueError("Legacy comparison must use registered finalepoch20 predictions")
            with legacy.open()as stream:legacy_rows=list(csv.DictReader(stream))
            legacy_ids=[row["sample_id"]for row in legacy_rows]
            _verify_record(evidence[model_id]["legacy_predictions"],root)
            if [int(row["label"])for row in legacy_rows]!=dev["labels"].tolist():raise ValueError("Legacy source labels differ from fixed development labels")
            if legacy_ids!=dev["sample_ids"]:raise ValueError("Legacy source prediction membership differs")
            legacy_score=100*sum(int(row["label"])==int(row["top1_prediction"])for row in legacy_rows)/len(legacy_rows)
            if abs(legacy_score-100*old["probe"]["summary"]["metrics"]["top1"])>1e-4:raise ValueError("Legacy score disagrees with source CSV")
            arm={"id":old["arm_id"],"seed":old["seed"],"variant":old["variant"],"kind":old["kind"]}
            try:value=_evaluate_caches(root,artifact,root/"checkpoints"/checkpoint_run/"convex_cached14"/model_id,model_id,arm,"final",600,train,dev,old["source_encoder"],config,deadline_unix,old["training_cost"])
            except TimeoutError:break
            value.update(legacy_epoch20_accuracy_percent=legacy_score,legacy_predictions=_record(legacy),cached_identity_hash=_hash(identity),scope="Cached actual final model; own-init learning UNKNOWN")
            atomic_json(recordpath,value);evaluations[model_id]=value
            atomic_json(artifact/"status.json",{"status":"RUNNING","completed_models":sum(v.get("converged",False)for v in evaluations.values()),"total_models":16,"updated_unix":time.time()})
            if value.get("status")!="COMPLETE":break
    complete=len(evaluations)==16 and all(v.get("converged")and v.get("knn",{}).get("status")=="COMPLETE"for v in evaluations.values())
    result={"schema":"notebook15-cached14-readout-results/v1","status":"COMPLETE"if complete else"UNKNOWN_INCOMPLETE_OR_NONCONVERGED","evaluations":evaluations,"rows":[{k:v for k,v in value.items()if k in("model_id","arm_id","seed","variant","endpoint","accuracy_percent","knn_accuracy_percent","converged","numerical_valid","probe_receipt","source_encoder","legacy_epoch20_accuracy_percent")}for value in evaluations.values()],
        "matched_initialization_scope":"NOT_AVAILABLE_FOR_THIS_ACTUAL_FINAL_CACHE_STUDY","new_jepa_training_updates":0,"official_test":"NOT_RUN","probe_configuration":config,"source":_record(source_path),"checkpoint_run_id":checkpoint_run}
    if complete:
        summary=[]
        for variant in sorted({v["variant"]for v in evaluations.values()}):
            group=[v for v in evaluations.values()if v["variant"]==variant];summary.append({"variant":variant,"legacy_epoch20_mean_accuracy_percent":float(np.mean([v["legacy_epoch20_accuracy_percent"]for v in group])),"converged_mean_accuracy_percent":float(np.mean([v["accuracy_percent"]for v in group])),"knn_mean_accuracy_percent":float(np.mean([v["knn_accuracy_percent"]for v in group])),"mean_readout_change_pp":float(np.mean([v["accuracy_percent"]-v["legacy_epoch20_accuracy_percent"]for v in group]))})
        result["readout_stability_comparison"]=summary
    atomic_json(artifact/"cached14_results.json",result);return result


In [ ]:
def score_converged_model(query_cache,training_probe_record,root=None):
    """Apply a certified training readout without fitting. Caller must seal official access before extracting query_cache."""
    root=Path(root or ROOT).resolve()
    receipt=_read(_verify_record(training_probe_record,root))if "path"in training_probe_record else training_probe_record
    if receipt.get("status")!="CONVERGED"or not receipt.get("converged"):raise ValueError("A converged training-only probe receipt is required")
    _verify_record(receipt["manifest"],root);checkpoint=_verify_record(receipt["latest_checkpoint"],root)
    state,manifest=_load_probe(checkpoint.parent)
    if manifest["latest"]!=checkpoint.name or _hash(state["identity"])!=receipt["identity_hash"]or not state["converged"]:raise ValueError("Certified readout state differs from receipt")
    cfg=probe_configuration(state["identity"]["config"]);last=state["metrics"][-1]
    if state["stable_checks"]<cfg["stable_checks"]or last["gradient_inf_norm"]>cfg["gradient_inf_tolerance"]or last["relative_objective_change"]>cfg["relative_objective_tolerance"]:raise ValueError("Readout training certificate no longer valid")
    features,labels,ids=query_cache["features"],query_cache["labels"],query_cache["sample_ids"]
    if features.ndim!=2 or len(features)!=len(labels)or len(ids)!=len(features)or len(set(ids))!=len(ids)or not len(ids)or not torch.isfinite(features).all():raise ValueError("Invalid query cache")
    if features.device.type!="cpu"or labels.device.type!="cpu"or labels.dtype!=torch.long or bool(((labels<0)|(labels>=int(cfg["num_classes"]))).any()):raise ValueError("Invalid query labels or device")
    if set(ids)&set(state["sampler"]["sample_ids"]):raise ValueError("Query/training membership leakage")
    if features.shape[1]!=state["model"]["weight"].shape[1]:raise ValueError("Readout feature dimensions differ")
    with torch.inference_mode():
        z=(features.double()-state["standardization"]["mean"])/state["standardization"]["std"]
        logits=torch.nn.functional.linear(z,state["model"]["weight"],state["model"]["bias"]);pred=logits.argmax(-1)
    if not torch.isfinite(logits).all():raise FloatingPointError("Nonfinite certified readout logits")
    return {"status":"COMPLETE","converged":True,"accuracy_percent":100*float((pred==labels).double().mean()),"examples":len(ids),
        "predictions":[{"sample_id":sid,"label":int(label),"top1_prediction":int(value)}for sid,label,value in zip(ids,labels.tolist(),pred.tolist())],
        "training_probe_record":receipt.get("receipt",training_probe_record),"latest_checkpoint":receipt["latest_checkpoint"],"selection":"Previously certified training objective; zero additional optimizer steps"}


def _exact_tree(a,b):
    if isinstance(a,torch.Tensor):return isinstance(b,torch.Tensor)and a.dtype==b.dtype and a.shape==b.shape and torch.equal(a,b)
    if isinstance(a,np.ndarray):return isinstance(b,np.ndarray)and np.array_equal(a,b)
    if isinstance(a,dict):return isinstance(b,dict)and set(a)==set(b)and all(_exact_tree(a[k],b[k])for k in a)
    if isinstance(a,(list,tuple)):return type(a)==type(b)and len(a)==len(b)and all(_exact_tree(x,y)for x,y in zip(a,b))
    return a==b


def fastprobe_smoke(root=None,artifact=None):
    """Small CPU SOFTWARE FIXTURE: exact4 vs2+2 LBFGS/RNG and known convex optimum, never scientific data."""
    from scipy.optimize import brentq
    root=Path(root or ROOT).resolve();artifact=Path(artifact or root/"artifacts/notebook_15_validation/probe_smoke").resolve();artifact.mkdir(parents=True,exist_ok=True)
    if not artifact.is_relative_to(root):raise ValueError("Software receipt must remain under project")
    run_id="n15-evaluation-software-"+str(time.time_ns());base=root/"checkpoints"/run_id;checks=[]
    def check(name,condition):
        if not condition:raise AssertionError(name)
        checks.append({"name":name,"status":"PASS"})
    generator=torch.Generator().manual_seed(81)
    train={"features":torch.randn(72,6,generator=generator),"labels":torch.arange(72)%3,"sample_ids":["software-train-"+str(i)for i in range(72)]}
    query={"features":torch.randn(24,6,generator=generator),"labels":torch.arange(24)%3,"sample_ids":["software-query-"+str(i)for i in range(24)]}
    cfg=probe_configuration({"num_classes":3,"l2":.1,"max_steps":100,"gradient_inf_tolerance":1e-8,"relative_objective_tolerance":1e-12,"checkpoint_every":2})
    caller=_rng();full=fit_convex_probe(train,query,base/"full",cfg,"exact-probe",stop_after_steps=4)
    check("caller all RNG unchanged",_exact_tree(caller,_rng()))
    part=fit_convex_probe(train,query,base/"split",cfg,"exact-probe",stop_after_steps=2)
    check("interruption committed step2",part["optimizer_steps"]==2 and not part["converged"])
    resumed=fit_convex_probe(train,query,base/"split",cfg,"exact-probe",stop_after_steps=4)
    a,am=_load_probe(base/"full");b,bm=_load_probe(base/"split")
    for key in ("model","optimizer","scheduler","scaler","standardization","rng","sampler","step","metrics","stable_checks","function_evaluations","converged","identity"):
        check("exact4 vs2+2 "+key,_exact_tree(a[key],b[key]))
    check("retain latest2 and best",len(am["checkpoints"])>=2 and am["best"]in{r["file"]for r in am["checkpoints"]})
    exhausted=fit_convex_probe(train,query,base/"exhausted",{**cfg,"max_steps":1},"nonconverged")
    check("unconverged is UNKNOWN with no score",exhausted["status"]=="UNKNOWN_NOT_CONVERGED"and exhausted["query_metrics"]is None and exhausted["predictions"]is None)
    cutoff=fit_convex_probe(train,query,base/"cutoff",cfg,"cutoff",deadline_unix=0.)
    check("cutoff is UNKNOWN with checkpoint",cutoff["status"]=="UNKNOWN_CUTOFF"and cutoff["query_metrics"]is None and cutoff["latest_checkpoint"]["sha256"])
    analytic_train={"features":torch.tensor([[-1.],[1.],[-1.],[1.]]),"labels":torch.tensor([0,1,0,1]),"sample_ids":["analytic-train-"+str(i)for i in range(4)]}
    analytic_query={"features":torch.tensor([[-1.],[1.]]),"labels":torch.tensor([0,1]),"sample_ids":["analytic-query-0","analytic-query-1"]}
    acfg=probe_configuration({"num_classes":2,"l2":.1,"max_steps":100,"gradient_inf_tolerance":1e-7,"relative_objective_tolerance":1e-10,"checkpoint_every":2})
    optimum=fit_convex_probe(analytic_train,analytic_query,base/"analytic",acfg,"analytic-optimum");state,manifest=_load_probe(base/"analytic")
    alpha=brentq(lambda x:.1*x-1/(1+math.exp(2*x)),0,10);expected_objective=math.log1p(math.exp(-2*alpha))+.1*alpha*alpha
    check("known strictly regularized symmetric optimum converges",optimum["converged"])
    check("independent scalar optimum objective",abs(optimum["train_objective"]-expected_objective)<1e-10)
    check("independent scalar optimum weights",torch.allclose(state["model"]["weight"],torch.tensor([[-alpha],[alpha]],dtype=torch.float64),atol=1e-6,rtol=0))
    check("training certificate meets fixed tolerances",optimum["gradient_inf_norm"]<=acfg["gradient_inf_tolerance"]and optimum["relative_objective_change"]<=acfg["relative_objective_tolerance"]and optimum["stable_checks"]>=acfg["stable_checks"])
    score=score_converged_model(analytic_query,optimum["receipt"],root)
    check("read-only application equals certified prediction",score["accuracy_percent"]==100 and score["predictions"]==optimum["predictions"])
    result={"status":"PASS","evidence_role":"SYNTHETIC_SOFTWARE_VALIDATION_ONLY_NO_RESEARCH_RESULTS","run_id":run_id,"checks":checks,"check_count":len(checks),"evaluation_export_hash":export_source_hash(),"checkpoint_root":str(base),"independent_optimum":alpha}
    atomic_json(artifact/"probe_smoke.json",result);return result


In [ ]:
def _seal_verifier(root):
    path=Path(root)/"experiments/16_hierarchical_confirmation/confirmation.ipynb";name="fmm16_evaluation_seal_verifier"
    if name not in sys.modules:
        module=types.ModuleType(name);module.__file__=str(path);TaggedNotebookLoader(path).exec_module(module);sys.modules[name]=module
    return sys.modules[name].verify_seal


def evaluate_model_on_ids(root,artifact,config,source_encoder_record,train_ids,test_ids,dataset,training_probe_record=None,deadline_unix=None,seal_record=None):
    """Sealed official readout. Verify immutable seal before any dataset read; fit/selection remain training-only."""
    root,artifact=Path(root).resolve(),Path(artifact).resolve();train_ids,test_ids=list(train_ids),list(test_ids)
    if not artifact.is_relative_to(root)or not train_ids or not test_ids or len(set(train_ids))!=len(train_ids)or len(set(test_ids))!=len(test_ids)or set(train_ids)&set(test_ids):raise ValueError("Valid separated unique sealed memberships and project artifact required")
    if seal_record is None:raise ValueError("Official test access requires an immutable registered seal")
    _verify_record(source_encoder_record,root);_deadline(deadline_unix)
    seal=_read(_verify_record(seal_record,root))if "path"in seal_record else seal_record
    # This validator checks registration/source/model/membership, without fetching images.
    seal_validation=_seal_verifier(root)(root,artifact,seal,test_ids,source_encoder_record)
    if seal_validation is False:raise ValueError("Official seal verification failed")
    builder,builder_identity=_builder(root);configuration=probe_configuration(config.get("probe_convex"));variant=config.get("kernel15",{}).get("variant",config.get("variant","candidate"));seed=int(config.get("seeds",{}).get("model_init",0))
    if isinstance(seal_validation,dict):
        sources=builder_identity.get("source_hashes",builder_identity)
        if seal_validation.get("source_hashes")!=sources:raise ValueError("Official readout builder differs from sealed registration")
        if "probe_configuration"not in seal_validation or probe_configuration(seal_validation["probe_configuration"])!=configuration:raise ValueError("Official probe differs from frozen registration")
        if seal_validation.get("probe_training_ids_sha256")!=_hash(train_ids):raise ValueError("Official probe train membership differs from registration")
    kind="vit"if variant=="vit"else"semantic";model=builder(copy.deepcopy(config),kind).cpu();state=torch.load(source_encoder_record["path"],map_location="cpu",weights_only=False)
    if isinstance(state,dict)and "model"in state:
        if {k:v for k,v in state.get("config",{}).items()if k!="probe_convex"}!={k:v for k,v in config.items()if k!="probe_convex"}:raise ValueError("Sealed encoder checkpoint config differs")
        model.load_state_dict(state["model"])
    else:model.target.load_state_dict(state)
    encoder=model.target;del model,state
    model_id="s"+str(seed)+"-"+str(variant)+"-"+source_encoder_record["sha256"][:12]
    folder=artifact/"final_evaluation"/model_id;folder.mkdir(parents=True,exist_ok=True)
    identity={"schema":"notebook16-sealed-convex-readout/v1","config":_canonical(config),"source_encoder":source_encoder_record,"builder":builder_identity,"seal_hash":_hash(seal),"training_ids":train_ids,"heldout_ids":test_ids,"dataset_fingerprint":dataset.fingerprint,"probe_config":configuration,"evaluation_export_hash":export_source_hash()}
    binding=folder/"registration.json"
    if binding.exists():
        if _read(binding)!=identity:raise ValueError("Sealed readout source/config/protocol identity changed")
    else:atomic_json(binding,identity)
    with run_writer(folder/"writer_lock"):
        common={"dataset_revision":dataset.fingerprint,"checkpoint_hash":encoder_state_hash(encoder),"encoder_kind":getattr(encoder,"kind",kind),"preprocessing":preprocessing_descriptor(config),"pooling":"mean_final_layernorm_patches","normalization":"Final LayerNorm patch mean; probe standardization fit on training features only"}
        train=extract_features(encoder,_DeadlineDataset(EvaluationDataset(dataset,train_ids,config),deadline_unix),folder/"features",{**common,"sample_ids":train_ids},batch_size=int(config.get("evaluation",{}).get("batch_size",64)),device=config.get("device","cpu"))
        if training_probe_record is None:
            # No official/dev labels used while fitting or certifying this readout.
            placeholder={"features":train["features"][:1].clone(),"labels":torch.zeros(1,dtype=torch.long),"sample_ids":["__software_training_certificate_placeholder__"]}
            probe=fit_convex_probe(train,placeholder,root/"checkpoints"/seal_validation["run_id"]/"final_convex"/model_id,configuration,model_id,deadline_unix=deadline_unix)
            if not probe["converged"]:
                result={"status":"UNKNOWN_PROBE_NOT_CONVERGED","arm_id":model_id,"seed":seed,"variant":variant,"endpoint":"final","accuracy_percent":None,"knn_accuracy_percent":None,"converged":False,"numerical_valid":None,"probe_receipt":probe["receipt"],"heldout_status":"NOT_READ","heldout_membership_sha256":_hash(test_ids),"source_encoder":source_encoder_record};atomic_json(folder/"results.json",result);return result
            training_probe_record=probe["receipt"]
        receipt=_read(_verify_record(training_probe_record,root));_verify_record(receipt["latest_checkpoint"],root)
        probe_state,manifest=_load_probe(Path(receipt["latest_checkpoint"]["path"]).parent)
        if probe_state["identity"]["config"]!=configuration or probe_state["identity"]["training_sample_ids"]!=train_ids or probe_state["identity"]["training_content_hash"]!=encoder_state_hash({"features":train["features"],"labels":train["labels"]})or probe_state["identity"]["training_cache_identity"]!=train["identity"]:raise ValueError("Sealed training probe differs from encoder/cache/protocol")
        if not receipt.get("converged"):raise ValueError("Sealed readout requires a converged training probe")
        # Repeat seal verification immediately before first official-test pixel access.
        _seal_verifier(root)(root,artifact,seal,test_ids,source_encoder_record);_deadline(deadline_unix)
        query=extract_features(encoder,_DeadlineDataset(EvaluationDataset(dataset,test_ids,config),deadline_unix),folder/"features",{**common,"sample_ids":test_ids},batch_size=int(config.get("evaluation",{}).get("batch_size",64)),device=config.get("device","cpu"))
        encoder.cpu();del encoder
        score=score_converged_model(query,training_probe_record,root);knn=cosine_knn(train,query,configuration,deadline_unix);diagnostic=_diagnostics(train,query)
        valid=all(diagnostic[k]for k in("finite_train_and_development","training_representation_not_exactly_constant","development_representation_not_exactly_constant"))
        prediction_path=folder/"predictions.csv";write_csv(prediction_path,score["predictions"])
        result={"status":"COMPLETE"if knn.get("status")=="COMPLETE"else"UNKNOWN_KNN_CUTOFF","arm_id":model_id,"model_id":model_id,"seed":seed,"variant":variant,"endpoint":"final","accuracy_percent":score["accuracy_percent"],"knn_accuracy_percent":knn.get("accuracy_percent"),"converged":True,"numerical_valid":valid,"probe_receipt":training_probe_record,"source_encoder":source_encoder_record,"feature_caches":{"train":_record(train["path"]),"official_test":_record(query["path"])},"prediction_record":_record(prediction_path),"representation_diagnostics":diagnostic,"knn":{k:v for k,v in knn.items()if k!="predictions"},"heldout_status":"AUTHORIZED_FINAL_EVALUATION","heldout_membership_sha256":_hash(test_ids),"seal_sha256":_hash(seal),"readout_selection":"Certified fixed training objective only; no official-test tuning","evaluation_registration":_record(binding)}
        atomic_json(folder/"results.json",result);return result


In [ ]:
def evaluate_endpoint(root,artifact,arm,bundle,modelstate_record,actual_updates,deadline_unix,probe_config=None,endpoint="final",training_cost=None,run_id=None,stage_id="endpoints"):
    """One source-bound development endpoint, including unequal update counts under16equal-wall registration."""
    root,artifact=Path(root).resolve(),Path(artifact).resolve();step=int(actual_updates)
    if not artifact.is_relative_to(root)or endpoint not in("initial","final")or step<0 or(endpoint=="initial"and step!=0):raise ValueError("Valid project endpoint/count required")
    if arm["config"].get("evaluation",{}).get("heldout_enabled",False):raise ValueError("Development endpoint cannot enable official test")
    if(set(bundle["probe_train_ids"])|set(bundle["dev_ids"]))&set(bundle.get("heldout_ids",[]))or set(bundle["probe_train_ids"])&set(bundle["dev_ids"]):raise ValueError("Endpoint membership leakage")
    builder,builder_identity=_builder(root);configuration=probe_configuration(probe_config);_verify_record(modelstate_record,root);_deadline(deadline_unix)
    model_id=arm["id"]+("-initial"if endpoint=="initial"else"-u"+str(step));folder=artifact/"endpoints"/model_id;folder.mkdir(parents=True,exist_ok=True)
    identity={"schema":"notebook15-source-bound-single-endpoint/v1","arm":_canonical(arm),"endpoint":endpoint,"actual_successful_updates":step,"source_encoder":modelstate_record,"builder":builder_identity,"probe_configuration":configuration,"evaluation_export_hash":export_source_hash(),"dataset_fingerprint":bundle["dataset"].fingerprint,"training_ids":list(bundle["probe_train_ids"]),"development_ids":list(bundle["dev_ids"])}
    registration=folder/"registration.json"
    if registration.exists():
        if _read(registration)!=identity:raise ValueError("Endpoint registration/source/probe identity changed")
    else:atomic_json(registration,identity)
    result_path=folder/"evaluation.json"
    with run_writer(folder/"writer_lock"):
        if result_path.exists():
            saved=_read(result_path)
            if saved["endpoint_identity_hash"]!=_hash(identity):raise ValueError("Endpoint evidence identity differs")
            _verify_record(saved["source_encoder"],root);_verify_record(saved["probe_receipt"],root)
            for record in saved["feature_caches"].values():_cache_from_record(root,record)
            if saved.get("prediction_record"):_verify_record(saved["prediction_record"],root)
            if saved.get("status")=="COMPLETE":return saved
        model=builder(copy.deepcopy(arm["config"]),arm["kind"]).cpu();state=torch.load(modelstate_record["path"],map_location="cpu",weights_only=(endpoint=="initial"))
        if endpoint=="initial":model.target.load_state_dict(state)
        else:
            if int(state["state"]["successful_updates"])!=step or state["config"]!=arm["config"]:raise ValueError("Endpoint successful update count/config differs")
            model.load_state_dict(state["model"])
        encoder=model.target;del model,state
        common={"dataset_revision":bundle["dataset"].fingerprint,"checkpoint_hash":encoder_state_hash(encoder),"encoder_kind":getattr(encoder,"kind",arm["kind"]),"preprocessing":preprocessing_descriptor(arm["config"]),"pooling":"mean_final_layernorm_patches","normalization":"Final LayerNorm patch mean; probe standardization fit on training features only"};caches={}
        for split,ids in(("train",bundle["probe_train_ids"]),("development",bundle["dev_ids"])):
            caches[split]=extract_features(encoder,_DeadlineDataset(EvaluationDataset(bundle["dataset"],ids,arm["config"]),deadline_unix),folder/"features",{**common,"sample_ids":list(ids)},batch_size=int(arm["config"].get("evaluation",{}).get("batch_size",64)),device=arm["config"].get("device","cpu"))
        encoder.cpu();del encoder
        directory=root/"checkpoints"/(run_id or artifact.name)/"stages"/stage_id/"convex_probes"/model_id
        value=_evaluate_caches(root,folder,directory,model_id,arm,endpoint,step,caches["train"],caches["development"],modelstate_record,configuration,deadline_unix,training_cost)
        value.update(endpoint_identity_hash=_hash(identity),endpoint_registration=_record(registration));atomic_json(result_path,value);return value
